In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore") 
print("✅ Libraries loaded")
 

✅ Libraries loaded


In [2]:
raw_budget = {
    "year":       ["2018-19","2019-20","2020-21","2021-22","2022-23","2023-24","2024-25"],
    "budget_est": [ 2400,     6400,     6975,     6400,     7200,     7500,     9406   ],
    "revised_est":[ 2160,     3200,     3475,     3400,     7200,     6881,     9406   ],
    "utilised":   [ 1630,     2800,     3100,     3200,     6000,     6500,     3920   ],
}

In [3]:
raw_claims = {
    "year":    ["2019-20","2020-21","2021-22","2022-23"],
    "filed":   [   9.3,      8.1,    12.1,     15.4  ],
    "settled": [   8.8,      7.6,    11.5,     14.7  ],
    "rejected":[   0.5,      0.5,     0.6,      0.7  ],
}

In [4]:
raw_fraud = {
    "irregularity": [
        "Ghost registrations (invalid mobile)",
        "Dead patient claims filed",
        "Claims paid for dead patients",
        "Simultaneous hospitalisations",
        "Ineligible enrolments (J&K sample)",
        "Male patients with maternity claims",
    ],
    "count":    [749820, 214923, 3903, 78396, 17200, 23670],
    "severity": ["Critical","Critical","Critical","High","High","Medium"],
}

In [5]:
raw_oope = {
    "year":      ["FY15","FY16","FY17","FY18","FY19","FY20","FY21","FY22"],
    "oope_pct":  [ 62.6,  61.1,  59.8,  58.7,  57.0,  52.1,  47.1,  39.4],
    "govt_pct":  [ 29.0,  30.5,  32.1,  33.8,  35.5,  40.3,  45.1,  48.0],
    "pmjay_era": ["Pre","Pre","Pre","Pre","Post","Post","Post","Post"],
}
 

In [6]:
raw_states = {
    "state": [
        "Andhra Pradesh","Assam","Tamil Nadu","Rajasthan",
        "Himachal Pradesh","Chhattisgarh","Gujarat","Uttarakhand",
        "Jharkhand","Maharashtra","Kerala","Karnataka",
        "Madhya Pradesh","Uttar Pradesh","Punjab","Haryana"
    ],
    "utilisation_pct": [72,71,68,64,67,61,58,58,55,51,53,49,48,42,39,44],
    "region": [
        "South","East","South","North","North","Central","West","North",
        "East","West","South","South","Central","North","North","North"
    ],
}
 
print("✅ Raw data dictionaries created")

✅ Raw data dictionaries created


In [7]:
df_budget = pd.DataFrame(raw_budget)
df_budget["utilisation_rate"] = (
    df_budget["utilised"] / df_budget["budget_est"] * 100
).round(1)
df_budget["savings_gap"] = df_budget["budget_est"] - df_budget["utilised"]
 
print("── Budget DataFrame ──")
print(df_budget.to_string(index=False))
print()

── Budget DataFrame ──
   year  budget_est  revised_est  utilised  utilisation_rate  savings_gap
2018-19        2400         2160      1630              67.9          770
2019-20        6400         3200      2800              43.8         3600
2020-21        6975         3475      3100              44.4         3875
2021-22        6400         3400      3200              50.0         3200
2022-23        7200         7200      6000              83.3         1200
2023-24        7500         6881      6500              86.7         1000
2024-25        9406         9406      3920              41.7         5486



In [8]:
df_claims = pd.DataFrame(raw_claims)
df_claims["settlement_rate"] = (
    df_claims["settled"] / df_claims["filed"] * 100
).round(1)
df_claims["rejection_rate"] = (
    df_claims["rejected"] / df_claims["filed"] * 100
).round(1)
 
print("── Claims DataFrame ──")
print(df_claims.to_string(index=False))
print()

── Claims DataFrame ──
   year  filed  settled  rejected  settlement_rate  rejection_rate
2019-20    9.3      8.8       0.5             94.6             5.4
2020-21    8.1      7.6       0.5             93.8             6.2
2021-22   12.1     11.5       0.6             95.0             5.0
2022-23   15.4     14.7       0.7             95.5             4.5



In [9]:
df_fraud = pd.DataFrame(raw_fraud)
df_fraud["share_pct"] = (
    df_fraud["count"] / df_fraud["count"].sum() * 100
).round(1)
df_fraud = df_fraud.sort_values("count", ascending=True).reset_index(drop=True)
 
print("── Fraud DataFrame ──")
print(df_fraud.to_string(index=False))
print()

── Fraud DataFrame ──
                        irregularity  count severity  share_pct
       Claims paid for dead patients   3903 Critical        0.4
  Ineligible enrolments (J&K sample)  17200     High        1.6
 Male patients with maternity claims  23670   Medium        2.2
       Simultaneous hospitalisations  78396     High        7.2
           Dead patient claims filed 214923 Critical       19.8
Ghost registrations (invalid mobile) 749820 Critical       68.9



In [10]:
df_oope = pd.DataFrame(raw_oope)
df_oope["oope_change"] = df_oope["oope_pct"].diff().round(1)  # Year-on-year change
 
print("── OOPE DataFrame ──")
print(df_oope.to_string(index=False))
print()

── OOPE DataFrame ──
year  oope_pct  govt_pct pmjay_era  oope_change
FY15      62.6      29.0       Pre          NaN
FY16      61.1      30.5       Pre         -1.5
FY17      59.8      32.1       Pre         -1.3
FY18      58.7      33.8       Pre         -1.1
FY19      57.0      35.5      Post         -1.7
FY20      52.1      40.3      Post         -4.9
FY21      47.1      45.1      Post         -5.0
FY22      39.4      48.0      Post         -7.7



In [11]:
df_states = pd.DataFrame(raw_states)
df_states = df_states.sort_values("utilisation_pct").reset_index(drop=True)
df_states["category"] = df_states["utilisation_pct"].apply(
    lambda x: "Poor (<50%)" if x < 50 else ("Moderate (50-65%)" if x < 65 else "Good (>65%)")
)
 
print("── States DataFrame ──")
print(df_states.to_string(index=False))
 
print("\n✅ Preprocessing complete")
 

── States DataFrame ──
           state  utilisation_pct  region          category
          Punjab               39   North       Poor (<50%)
   Uttar Pradesh               42   North       Poor (<50%)
         Haryana               44   North       Poor (<50%)
  Madhya Pradesh               48 Central       Poor (<50%)
       Karnataka               49   South       Poor (<50%)
     Maharashtra               51    West Moderate (50-65%)
          Kerala               53   South Moderate (50-65%)
       Jharkhand               55    East Moderate (50-65%)
     Uttarakhand               58   North Moderate (50-65%)
         Gujarat               58    West Moderate (50-65%)
    Chhattisgarh               61 Central Moderate (50-65%)
       Rajasthan               64   North Moderate (50-65%)
Himachal Pradesh               67   North       Good (>65%)
      Tamil Nadu               68   South       Good (>65%)
           Assam               71    East       Good (>65%)
  Andhra Pradesh 

In [13]:
print("=" * 55)
print("  AYUSHMAN BHARAT — KEY STATISTICS")
print("=" * 55)
print(f"\n  Budget (2018-25):")
print(f"    Total allocated : ₹{df_budget['budget_est'].sum():,} Crore")
print(f"    Total utilised  : ₹{df_budget['utilised'].sum():,} Crore")
print(f"    Avg util. rate  : {df_budget['utilisation_rate'].mean():.1f}%")
print(f"    Best year       : {df_budget.loc[df_budget['utilisation_rate'].idxmax(),'year']} "
      f"({df_budget['utilisation_rate'].max():.0f}%)")
print(f"    Worst year      : {df_budget.loc[df_budget['utilisation_rate'].idxmin(),'year']} "
      f"({df_budget['utilisation_rate'].min():.0f}%)")

  AYUSHMAN BHARAT — KEY STATISTICS

  Budget (2018-25):
    Total allocated : ₹46,281 Crore
    Total utilised  : ₹27,150 Crore
    Avg util. rate  : 59.7%
    Best year       : 2023-24 (87%)
    Worst year      : 2024-25 (42%)


In [14]:
print(f"\n  Claims (2019-23):")
print(f"    Avg settlement rate : {df_claims['settlement_rate'].mean():.1f}%")
print(f"    Total claims filed  : {df_claims['filed'].sum():.1f}M")
print(f"    Total settled       : {df_claims['settled'].sum():.1f}M")
print(f"    Total rejected      : {df_claims['rejected'].sum():.1f}M")
 
print(f"\n  Fraud (CAG 2023):")
print(f"    Total cases mapped  : {df_fraud['count'].sum():,}")
print(f"    Most common issue   : {df_fraud.iloc[-1]['irregularity']}")
 


  Claims (2019-23):
    Avg settlement rate : 94.7%
    Total claims filed  : 44.9M
    Total settled       : 42.6M
    Total rejected      : 2.3M

  Fraud (CAG 2023):
    Total cases mapped  : 1,087,912
    Most common issue   : Ghost registrations (invalid mobile)


In [15]:
print(f"\n  OOPE trend:")
pre  = df_oope[df_oope["pmjay_era"]=="Pre"]["oope_pct"].mean()
post = df_oope[df_oope["pmjay_era"]=="Post"]["oope_pct"].mean()
print(f"    Avg OOPE pre-PMJAY  : {pre:.1f}%")
print(f"    Avg OOPE post-PMJAY : {post:.1f}%")
print(f"    Reduction           : {pre - post:.1f} pp (percentage points)")
 
print(f"\n  State performance:")
print(f"    States rated 'Poor'     : {(df_states['category']=='Poor (<50%)').sum()}")
print(f"    States rated 'Moderate' : {(df_states['category']=='Moderate (50-65%)').sum()}")
print(f"    States rated 'Good'     : {(df_states['category']=='Good (>65%)').sum()}")
print(f"    National avg util.      : {df_states['utilisation_pct'].mean():.1f}%")
 
print("\n✅ Summary statistics done")


  OOPE trend:
    Avg OOPE pre-PMJAY  : 60.5%
    Avg OOPE post-PMJAY : 48.9%
    Reduction           : 11.6 pp (percentage points)

  State performance:
    States rated 'Poor'     : 5
    States rated 'Moderate' : 7
    States rated 'Good'     : 4
    National avg util.      : 56.2%

✅ Summary statistics done


In [17]:
COLORS = {
    "red":    "#c0392b",
    "orange": "#e67e22",
    "gold":   "#c9a84c",
    "green":  "#27ae60",
    "dark":   "#2c2c2a",
    "muted":  "#7a7265",
    "bg":     "#f5f0e8",
}
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "figure.facecolor": COLORS["bg"],
    "axes.facecolor":   "#ffffff",
    "axes.edgecolor":   "#bfb9ad",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "grid.color":       "#ede8dc",
    "grid.linewidth":   0.8,
    "xtick.color":      COLORS["muted"],
    "ytick.color":      COLORS["muted"],
    "axes.titlepad":    12,
})
 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    "Ayushman Bharat — Data Analysis Dashboard",
    fontsize=15, fontweight="bold", y=0.98, x=0.04, ha="left", color=COLORS["dark"]
)
 
ax1 = axes[0, 0]
x   = np.arange(len(df_budget))
w   = 0.30
 
ax1.bar(x - w, df_budget["budget_est"], w, label="Budget Est.",  color=COLORS["dark"],  alpha=0.85)
ax1.bar(x,     df_budget["revised_est"],w, label="Revised Est.", color=COLORS["gold"],  alpha=0.85)
ax1.bar(x + w, df_budget["utilised"],   w, label="Utilised",     color=COLORS["red"],   alpha=0.85)
 
ax1.set_xticks(x)
ax1.set_xticklabels(df_budget["year"], rotation=35, ha="right", fontsize=8)
ax1.set_ylabel("₹ Crore", fontsize=9)
ax1.set_title("Budget vs Utilisation (₹ Crore)", fontsize=11, fontweight="bold")
ax1.legend(fontsize=8, framealpha=0)
ax1.grid(axis="y")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"₹{v/1000:.0f}K"))
ax1.annotate(
    "Source: MoHFW Parliamentary replies, Feb 2024",
    xy=(0, -0.22), xycoords="axes fraction",
    fontsize=7.5, color=COLORS["muted"], style="italic"
)
 
#Chart 2: Fraud Findings////////////////////////////////////////////////////////////////
ax2 = axes[0, 1]
fraud_colors = [
    COLORS["red"] if s == "Critical" else
    COLORS["orange"] if s == "High" else
    COLORS["gold"]
    for s in df_fraud["severity"]
]
bars = ax2.barh(
    df_fraud["irregularity"], df_fraud["count"],
    color=fraud_colors, edgecolor="none", height=0.55
)
for bar, val in zip(bars, df_fraud["count"]):
    ax2.text(
        bar.get_width() + 3000, bar.get_y() + bar.get_height() / 2,
        f"{val:,}", va="center", fontsize=8
    )
ax2.set_xlim(0, df_fraud["count"].max() * 1.25)
ax2.set_title("CAG Audit — Fraud Irregularities", fontsize=11, fontweight="bold")
legend_patches = [
    mpatches.Patch(color=COLORS["red"],    label="Critical"),
    mpatches.Patch(color=COLORS["orange"], label="High"),
    mpatches.Patch(color=COLORS["gold"],   label="Medium"),
]
ax2.legend(handles=legend_patches, fontsize=8, framealpha=0)
ax2.grid(axis="x")
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"{v/1e5:.1f}L" if v >= 1e5 else f"{v/1000:.0f}K"))
ax2.annotate(
    "Source: CAG Report No. 11 of 2023",
    xy=(0, -0.12), xycoords="axes fraction",
    fontsize=7.5, color=COLORS["muted"], style="italic"
)
 
#Chart 3: OOPE Trend////////////////////////////////////////////////////////////////////////////////////
ax3 = axes[1, 0]
ax3.plot(df_oope["year"], df_oope["oope_pct"],  "o-",
         color=COLORS["red"],   lw=2.5, ms=6, label="OOPE %")
ax3.plot(df_oope["year"], df_oope["govt_pct"],  "s-",
         color=COLORS["green"], lw=2.5, ms=6, label="Govt Health Exp %")
 
# Shade post-PMJAY
pmjay_idx = df_oope[df_oope["pmjay_era"] == "Post"].index[0]
ax3.axvspan(pmjay_idx - 0.5, len(df_oope) - 0.5,
            alpha=0.06, color=COLORS["green"], zorder=0)
ax3.text(pmjay_idx - 0.3, 68, "PMJAY →", fontsize=8, color=COLORS["muted"])
 
ax3.set_ylim(20, 75)
ax3.set_xticklabels(df_oope["year"], rotation=20, ha="right")
ax3.set_ylabel("%", fontsize=9)
ax3.set_title("Out-of-Pocket Expenditure vs Govt Health Spend (%)", fontsize=11, fontweight="bold")
ax3.legend(fontsize=8, framealpha=0)
ax3.grid(axis="y")
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"{v:.0f}%"))
ax3.annotate(
    "Source: Economic Survey 2024-25; NHA Health Accounts",
    xy=(0, -0.12), xycoords="axes fraction",
    fontsize=7.5, color=COLORS["muted"], style="italic"
)
 
#Chart 4: State Utilisation////////////////////////////////////////////////////////////////////////////////////////
ax4 = axes[1, 1]
state_colors = [
    COLORS["red"]    if c == "Poor (<50%)" else
    COLORS["orange"] if c == "Moderate (50-65%)" else
    COLORS["green"]
    for c in df_states["category"]
]
ax4.barh(df_states["state"], df_states["utilisation_pct"],
         color=state_colors, edgecolor="none", height=0.65)
ax4.axvline(50, ls="--", lw=1.2, color=COLORS["muted"], label="50% line")
ax4.axvline(65, ls=":",  lw=1.2, color=COLORS["green"],  label="65% target")
for i, row in df_states.iterrows():
    ax4.text(row["utilisation_pct"] + 0.5, i,
             f'{row["utilisation_pct"]}%', va="center", fontsize=7.5)
ax4.set_xlim(0, 85)
ax4.set_title("State-wise Estimated Utilisation (%)", fontsize=11, fontweight="bold")
ax4.legend(fontsize=8, framealpha=0, loc="lower right")
ax4.grid(axis="x")
ax4.annotate(
    "Source: Estimated from NHA, state reports & literature",
    xy=(0, -0.08), xycoords="axes fraction",
    fontsize=7.5, color=COLORS["muted"], style="italic"
)
 
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("ayushman_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Charts saved as ayushman_charts.png")
 